# Init Lakehouse

In [1]:
%%configure -f
{
    "defaultLakehouse": {"name": "DE_LH_100_BondedWarehouse"}
}

StatementMeta(, 7c06dc9f-7260-418f-b515-5d4ef764ff94, -1, Finished, Available, Finished)

# Init Imports (these need cutting-down post creation)

In [2]:
import os
import csv
import re
import shutil
import unicodedata
import pandas as pd

import notebookutils

#from decimal import Decimal
from datetime import datetime
from datetime import timedelta
#from collections import Counter
#from functools import reduce
import time

#from pyspark import StorageLevel
from pyspark.sql import DataFrame, Row
from pyspark.sql.functions import col, lit, when, concat, concat_ws, coalesce, count, monotonically_increasing_id, sum, to_date, udf, current_timestamp, length, substring, split, size, asc, row_number, desc, trim, regexp_replace
from pyspark.sql.functions import broadcast, hash, array, expr, array_distinct, date_format
from pyspark.sql.types import *
#from pyspark.sql import Window
from pyspark.sql import functions as F
from delta.tables import DeltaTable

StatementMeta(, 7c06dc9f-7260-418f-b515-5d4ef764ff94, 3, Finished, Available, Finished)

# Init Export Process

In [3]:
def save_dataframe_to_csv(df, file_path, show_header=False, mode='overwrite'):
    """
    Save a DataFrame as a single CSV file in a PySpark application.

    Parameters:
    df (pyspark.sql.DataFrame): The DataFrame to save.
    file_path (str): The path to save the CSV file.
    header (bool): Whether to include the header in the CSV file. Default is True.
    mode (str): The write mode. Options are 'overwrite', 'append', 'ignore', 'error' or 'errorifexists'. Default is 'overwrite'.

    Returns:
    None
    """

    use_pipes = len(df.columns) != 1
    print(f'Add pipes: {use_pipes}')
    print(f'Show headers: {show_header}')

    pandas_df = df.toPandas()
    
    # Replace newlines and carriage returns
    pandas_df = pandas_df.replace({r'\r\n': ' ', r'\n': ' ', r'\r': ' '}, regex=True)

    # Create a string representation of the DataFrame with '|' as separator
    # Escape special characters such as commas and pipes
    if use_pipes:
        csv_data = pandas_df.to_csv(sep="|", index=False, header=show_header, quoting=csv.QUOTE_NONE, escapechar="\\")
    else:
        csv_data = pandas_df.to_csv(sep="~", index=False, header=show_header, quoting=csv.QUOTE_NONE, escapechar="\\")

    # Add trailing pipe '|' at the end of each line
    csv_data_with_pipe = '\n'.join([line + '|' for line in csv_data.split('\n') if line])

    # Write to the file
    with open(file_path, 'w') as f:
        f.write(csv_data_with_pipe)


StatementMeta(, 7c06dc9f-7260-418f-b515-5d4ef764ff94, 4, Finished, Available, Finished)

# Init Debug & Incremental Vars

In [4]:
workspace_name = notebookutils.mssparkutils.env.getWorkspaceName()

if "DEV" in workspace_name.upper():
    debug = True
    incremental_run = False
    default_days_lag: int = 7

elif "UAT" in workspace_name.upper():
    debug = True
    incremental_run = True
    default_days_lag: int = 7

else:
    debug = False
    incremental_run = True
    default_days_lag: int = 0

if debug:
    print(debug , incremental_run)

StatementMeta(, 7c06dc9f-7260-418f-b515-5d4ef764ff94, 5, Finished, Available, Finished)

True False


# Init Days Lag Var

In [5]:
filterdate_pipe = ''

#default_days_lag: int = 1

enable_string_truncation = True
create_hash_cols: bool = False
transfer_file: bool = False
retain_error_records_in_ouput_file: bool = False

# Override Debug

#debug = False   #<<<<<<<<<<<<<<<<<<<<<<<<<<<  <<<<<<<<<<<<<<<<<<<<
#debug = True   #<<<<<<<<<<<<<<<<<<<<<<<<<<<  <<<<<<<<<<<<<<<<<<<<

# Override Full Run 

#incremental_run = False    #<<<<<<<<<<<<<<<<<<<<<<<<<<<  <<<<<<<<<<<<<<<<<<<<
#incremental_run = True     #<<<<<<<<<<<<<<<<<<<<<<<<<<<  <<<<<<<<<<<<<<<<<<<<

StatementMeta(, 7c06dc9f-7260-418f-b515-5d4ef764ff94, 6, Finished, Available, Finished)

In [6]:
filterdate = datetime.now() - timedelta(days=default_days_lag)
filterdate = filterdate.date()

if debug:
    print(f'Get Suppliers from: {filterdate}')

StatementMeta(, 7c06dc9f-7260-418f-b515-5d4ef764ff94, 7, Finished, Available, Finished)

Get Suppliers from: 2025-05-07


# Init Query(s)

In [7]:
suppliers_df = spark.sql(f"""
SELECT
	'ENDCTG' AS Company_Code
  ,vendtable.accountnum AS Supplier_Code
  ,dirpartytable.name AS Supplier_Name
  ,IFNULL(countrylookup.isocode2, '') AS Country_Consigned
  ,IFNULL(logisticspostaladdress.street, 'N/A') AS Street
  ,IFNULL(logisticspostaladdress.city, 'N/A') AS City
  ,IFNULL(logisticspostaladdress.zipcode, 'N/A') AS Post_Code
  ,logisticsaddresscountryregion.isocode AS Country
  ,'' AS C109_Ref 
  ,'' AS C109_Date 
  ,trandetermslookup.LangdonCode AS Trade_Terms
  ,'' AS Identification_Number
  
FROM vendtable

INNER JOIN dirpartytable
  ON vendtable.party = dirpartytable.recid

LEFT JOIN logisticspostaladdress
  ON dirpartytable.primaryaddresslocation = logisticspostaladdress.location

LEFT JOIN logisticsaddresscountryregion
  ON logisticspostaladdress.countryregionid = logisticsaddresscountryregion.countryregionid

LEFT JOIN trandetermslookup
  ON vendtable.dlvterm = trandetermslookup.Deliveryterms

LEFT JOIN countrylookup
  ON vendtable.hslsuppliercountryregionid = countrylookup.isocode3

WHERE
    vendtable.vendgroup IN ('Stock', 'Proforma', 'SDStock', 'Interco')
AND
    vendtable.dataareaid IN ('end.', 'END.')
AND
    DATE(logisticspostaladdress.validfrom) <= CURRENT_DATE
AND
    DATE(logisticspostaladdress.validto) > CURRENT_DATE

AND 
  GREATEST(
	vendtable.modifieddatetime, 
	dirpartytable.modifieddatetime, 
	logisticspostaladdress.modifieddatetime
	) >= '{filterdate}'

"""
)
if debug:
      display(suppliers_df)

StatementMeta(, 7c06dc9f-7260-418f-b515-5d4ef764ff94, 8, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 1e4b3a24-50d1-4e6f-8851-45c57bb1723b)

In [8]:
suppliers_df = suppliers_df.select(
    substring(col("Company_Code").cast("string"),1, 10).alias("Company_Code"),
    substring(col("Supplier_Code").cast("string"),1, 12).alias("Supplier_Code"),
    substring(col("Supplier_Name").cast("string"),1, 35).alias("Supplier_Name"),
    substring(col("Country_Consigned").cast("string"),1, 3).alias("Country_Consigned"),
    substring(col("Street").cast("string"),1, 35).alias("Street"),
    substring(col("City").cast("string"),1, 35).alias("City"),
    substring(col("Post_Code").cast("string"),1, 9).alias("Post_Code"),
    substring(col("Country").cast("string"),1, 3).alias("Country"),
    col("C109_Ref").cast("string").alias("C109_Ref"),
    col("C109_Date").cast("Date").alias("C109_Date"),
    substring(col("Trade_Terms").cast("string"),1, 17).alias("Trade_Terms"),
    col("Identification_Number").cast("string").alias("Identification_Number")
)
if debug:
    display(suppliers_df)

StatementMeta(, 7c06dc9f-7260-418f-b515-5d4ef764ff94, 9, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 1978e0ca-33e5-46d6-b3df-27aecac57d91)

# N/A to NA

In [9]:
suppliers_df = suppliers_df.replace("N/A", "NA")

StatementMeta(, 7c06dc9f-7260-418f-b515-5d4ef764ff94, 10, Finished, Available, Finished)

# Regex Pass to Remove Non-ASCII

In [10]:
for col in ["Street","City","Post_Code"]:

    suppliers_df = suppliers_df.withColumn(col, trim(regexp_replace(regexp_replace(col, "[^\\x00-\\x7F]|[\\|:#/?,Â%(),.-]", ""), "\\s+", " ")))

if debug:
    display(suppliers_df)

StatementMeta(, 7c06dc9f-7260-418f-b515-5d4ef764ff94, 11, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 3ae145dc-2a3f-4de9-8146-3d9715f39d25)

## Date Field Changing Post Query(s)

In [11]:
list_date_columns_1 = [name for name, dtype in suppliers_df.dtypes if dtype in ('date','timestamp')]

if debug:
    print("Date Columns to change: " , list_date_columns_1)

StatementMeta(, 7c06dc9f-7260-418f-b515-5d4ef764ff94, 12, Finished, Available, Finished)

Date Columns to change:  ['C109_Date']


In [12]:
for column in list_date_columns_1:
    suppliers_df = suppliers_df.withColumn(column, date_format(column, "dd-MM-yyyy"))

StatementMeta(, 7c06dc9f-7260-418f-b515-5d4ef764ff94, 13, Finished, Available, Finished)

In [13]:
if debug:
    display(suppliers_df)

StatementMeta(, 7c06dc9f-7260-418f-b515-5d4ef764ff94, 14, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 6bf9c874-427a-47e6-98ac-862c04764a32)

# Init Good File Name & Date Logic

In [14]:
file_path_folder = "/lakehouse/default/Files/Output/"
file_extention = '.dat'

# file date time - stamp tomorrow's date if after 6.15pm --Nick: had to knock it back 1 hour to account for timezone difference; working
now = datetime.now()
cutoff_time = now.replace(hour=17, minute=15, second=0, microsecond=0)

if now > cutoff_time:
    tomorrow = now + timedelta(days=1)
    #file_datetime = tomorrow.strftime('%Y-%m-%d')
    file_datetime = tomorrow.strftime('%Y-%m-%d-%H')
else:
    #file_datetime = now.strftime('%Y-%m-%d')
    file_datetime = now.strftime('%Y-%m-%d-%H')


file_name = "suppliers" + "_" + file_datetime + file_extention
file_path = file_path_folder + file_name

StatementMeta(, 7c06dc9f-7260-418f-b515-5d4ef764ff94, 15, Finished, Available, Finished)

In [15]:
if debug:
    print("Now: " , now)
    print("Cutoff: " , cutoff_time)
    print("File Date: " , file_datetime)
    print("File Folder Path: " , file_path_folder)
    print("Good File Name: " , file_name)
    print("Good File Name: " , file_path)

StatementMeta(, 7c06dc9f-7260-418f-b515-5d4ef764ff94, 16, Finished, Available, Finished)

Now:  2025-05-14 11:32:19.908015
Cutoff:  2025-05-14 17:15:00
File Date:  2025-05-14-11
File Folder Path:  /lakehouse/default/Files/Output/
Good File Name:  suppliers_2025-05-14-11.dat
Good File Name:  /lakehouse/default/Files/Output/suppliers_2025-05-14-11.dat


# Remove Rows Already Sent

In [16]:
if incremental_run:
    
    # REMOVE ROWS FROM CURRENT RUN THAT HAVE ALREADY BEEN SENT (EXIST IN RECORD TRACKING)

    lakehouse_table_name = "bondedwarehouserecordtracking_suppliers"
    container_column = "Supplier_Code"

    try:
        # Loads already sent Containers from record tracking
        sentrecords_df = spark.read.table(lakehouse_table_name).select(container_column).distinct()

        # Filter each input dataframe to EXCLUDE already sent
        suppliers_df = suppliers_df.join(sentrecords_df, suppliers_df["Supplier_Code"] == sentrecords_df["Supplier_Code"], "left_anti")

    except Exception as e:
        print(f"An error occurred: {e}")

StatementMeta(, 7c06dc9f-7260-418f-b515-5d4ef764ff94, 17, Finished, Available, Finished)

# Export Good

In [17]:
save_dataframe_to_csv(suppliers_df, file_path, show_header=False)

StatementMeta(, 7c06dc9f-7260-418f-b515-5d4ef764ff94, 18, Finished, Available, Finished)

Add pipes: True
Show headers: False


In [18]:
if suppliers_df.take(1):

    ready_to_copy = True

else:

    ready_to_copy = False

if debug:
    print(ready_to_copy)

StatementMeta(, 7c06dc9f-7260-418f-b515-5d4ef764ff94, 19, Finished, Available, Finished)

True


# Init Record Tracking

In [19]:
if incremental_run:

    # Save the final_df to different tables based on the exportfile variable value

    def record_tracking_df_to_table(dataframe, table_name, file_name):
        """Saves distinct records to a table, checking for duplicates and enabling column mapping."""
        table_name_lower = table_name.lower()

        # Check if table exists
        table_exists = True
        try:
            spark.read.table(table_name_lower)
            print(f"Table {table_name_lower} exists.")
        except Exception as e:
            print(f"Table {table_name_lower} does not exist.")
            table_exists = False

        # Columns to deduplicate on (excluding metadata)
        dedup_cols = [col for col in dataframe.columns if col not in ["Timestamp", "ExportName", "ExportDate"]]

        if table_exists:
            try:
                dataframe = dataframe.withColumn("ExportName", lit(file_name).cast(StringType())) \
                                    .withColumn("ExportDate", current_timestamp().cast(TimestampType()))

                existing_df = spark.read.table(table_name_lower)

                distinct_existing_df = existing_df.dropDuplicates(subset=dedup_cols)
                initial_existing_count = distinct_existing_df.count()
                print(f"Existing distinct count: {initial_existing_count}")

                distinct_new_df = dataframe.dropDuplicates(subset=dedup_cols)

                combined_distinct_df = distinct_new_df.unionByName(distinct_existing_df) \
                                                    .dropDuplicates(subset=dedup_cols)
                final_distinct_count = combined_distinct_df.count()
                print(f"Final distinct count: {final_distinct_count}")

                rows_added = final_distinct_count - initial_existing_count
                print(f"Added {rows_added} new distinct records to {table_name_lower}.")

                combined_distinct_df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name_lower)
                print(f"Distinct records saved to {table_name_lower}.")

            except Exception as e:
                print(f"Exception: Saving all records in new table. Exception: {e}")
                dataframe = dataframe.dropDuplicates(subset=dedup_cols)
                dataframe.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name_lower)
                print(f"Distinct records saved to {table_name_lower}.")

        else:
            try:
                print(f"Table doesn't exist. Saving all records in new table.")
                dataframe = dataframe.withColumn("ExportName", lit(file_name).cast(StringType())) \
                                    .withColumn("ExportDate", current_timestamp().cast(TimestampType()))
                dataframe = dataframe.dropDuplicates(subset=dedup_cols)
                dataframe.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name_lower)
                print(f"Distinct records saved to {table_name_lower}.")
            except Exception as e:
                print(f"Error saving data: {e}")

    table_prefix = 'BondedWarehouseRecordTracking_'

    record_tracking_df_to_table(suppliers_df, f"{table_prefix}suppliers", file_name)

StatementMeta(, 7c06dc9f-7260-418f-b515-5d4ef764ff94, 20, Finished, Available, Finished)

# Init Send To Azure Blob Storage

In [20]:
if ready_to_copy == False:
    output_msg = f'Process Complete'

    notebookutils.notebook.exit(output_msg)

StatementMeta(, 7c06dc9f-7260-418f-b515-5d4ef764ff94, 21, Finished, Available, Finished)

## Copy the file to an ADLS account for loading to the SFTP
Set the source and destination paths

In [21]:
if "DEV" in workspace_name.upper():
    dest_abfss_file_path = "Files/bonded_warehouse_dev/ToBeSent/" + file_name
    
elif "UAT" in workspace_name.upper():
    dest_abfss_file_path = "Files/bonded_warehouse_uat/ToBeSent/" + file_name

else:
    dest_abfss_file_path = "Files/bonded_warehouse/ToBeSent/" + file_name

StatementMeta(, 7c06dc9f-7260-418f-b515-5d4ef764ff94, 22, Finished, Available, Finished)

In [22]:
source_abfss_file_path = 'Files/Output/' + file_name

StatementMeta(, 7c06dc9f-7260-418f-b515-5d4ef764ff94, 23, Finished, Available, Finished)

In [23]:
if transfer_file:
    notebookutils.fs.fastcp(source_abfss_file_path, dest_abfss_file_path)

StatementMeta(, 7c06dc9f-7260-418f-b515-5d4ef764ff94, 24, Finished, Available, Finished)

In [24]:
if ready_to_copy == True:
    output_msg = f'Process Complete'

notebookutils.notebook.exit(output_msg)

StatementMeta(, 7c06dc9f-7260-418f-b515-5d4ef764ff94, 25, Finished, Available, Finished)

ExitValue: Process Complete